In [ ]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

In [ ]:
# Created LTM store + seed memories (done BEFORE running the graph)

store = InMemoryStore()

user_id = "u1"

user_details = ("user",user_id,"details")

store.put(user_details, "profile_1", {"data": "Name: Sorcerer Supreme Bhavya"})
store.put(user_details, "profile_2", {"data": "Profession: Computer science Student"})
store.put(user_details, "preference_1", {"data": "Prefers concise answers"})
store.put(user_details, "preference_2", {"data": "Likes examples in Python"})
store.put(user_details, "project_1", {"data": "Building GenAi Projects (Python-based project)"})

In [ ]:

# System prompt template 

SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user’s preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, Nitish...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP  server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible. For example, instead of "In TypeScript apps..." 
say "Since your project is built with TypeScript..."

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""


In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
def chatnode(state:MessagesState ,config:RunnableConfig,store:BaseStore):

    user_id = config['configurable']["user_id"]

    user_details = ("user",user_id,"details")
    
    # Getting all the details from the memory store
    items = store.search(user_details) 

    if items:
        user_details_content = "\n".join(f"-{i.value.get('data','')}" for i in items)
    else:
        user_details_content= ""

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        user_details_content=user_details_content
    )
    system_msg = SystemMessage(content= system_prompt)

    response = llm.invoke([system_msg] + state['messages'])
    return {"messgaes":[response]}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("chat", chatnode)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

graph = builder.compile(store=store)

graph